# 01. Filter Parking by Site Baseline / Post Windows

**User story:** As a Data Engineer, build a pipeline that filters time-series parking records by site-specific baseline and post-intervention bounds so processing volume is optimized and relevant.

### Why this notebook exists
- `sites_db.csv` defines **per-site** intervention timing (not one fixed global calendar window).
- City of Melbourne parking CSVs are very large; we keep only events that fall in each site's baseline/post window on matching streets.
- The reusable implementation lives in `src/ingestion/filter_parking_by_site_windows.py`. This notebook runs it and summarises coverage.


### Pipeline flow

![Time filter pipeline flow](time_filter_flow_chart.png)


### 1. Environment setup
Loads the project virtualenv packages and makes the repository root importable.


In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd().resolve()
root_dir = notebook_dir
for candidate in [notebook_dir, *notebook_dir.parents]:
    if (candidate / "config.yaml").exists() and (candidate / "src").exists():
        root_dir = candidate
        break

venv_site = root_dir / ".venv" / "Lib" / "site-packages"
if venv_site.exists() and str(venv_site) not in sys.path:
    sys.path.insert(0, str(venv_site))
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

print(f"Project root: {root_dir}")


### 2. Inspect configured site windows
Windows are derived from `sites_db.csv` using `config.yaml` → `processing.site_windows.window_months`.


In [ ]:
from src.config import SITE_WINDOW_MONTHS, SITES_DB_PATH, PARKING_YEARS, PROCESSED_DIR
from src.ingestion.filter_parking_by_site_windows import (
    build_site_windows,
    load_sites_db,
)

sites = load_sites_db(SITES_DB_PATH)
windows = build_site_windows(sites, window_months=SITE_WINDOW_MONTHS)

print(f"Window months: {SITE_WINDOW_MONTHS}")
print(f"Parking years: {PARKING_YEARS}")
print(f"Site/street windows: {len(windows):,}")
print(f"Distinct sites: {windows['SiteID'].nunique():,}")
windows.head(5)


### 3. Run the shared ingestion filter
This writes:
- `data/processed/site_parking_windows.parquet`
- `data/processed/parking_baseline.parquet`
- `data/processed/parking_post.parquet`

Expect this to take several minutes on the full 2013/2014 CSVs.


In [ ]:
from src.ingestion.filter_parking_by_site_windows import run_pipeline

run_pipeline()


### 5. Export results to CSV (Optional: easy to open)
Converts the processed parquet outputs into CSV files under `data/processed/` so you can open them in Excel or a text editor.

- `site_parking_windows.csv` — full window table (small)
- `parking_baseline.csv` — full baseline events
- `parking_post.csv` — full post events

Also writes small preview CSVs (`*_preview.csv`, first 5,000 rows) for quick viewing.


In [ ]:
import duckdb
from pathlib import Path

from src.config import PROCESSED_DIR

processed = Path(PROCESSED_DIR)
processed.mkdir(parents=True, exist_ok=True)

exports = [
    ("site_parking_windows.parquet", "site_parking_windows.csv", None),
    ("parking_baseline.parquet", "parking_baseline_preview.csv", 5000),
    ("parking_post.parquet", "parking_post_preview.csv", 5000),
]

con = duckdb.connect()
try:
    for parquet_name, csv_name, limit in exports:
        src = processed / parquet_name
        dest = processed / csv_name
        if not src.exists():
            print(f"Missing {src.name}; run the pipeline cell first.")
            continue

        src_posix = src.as_posix()
        dest_posix = dest.as_posix()
        if limit is None:
            query = f"""
            COPY (SELECT * FROM read_parquet('{src_posix}'))
            TO '{dest_posix}' (HEADER, DELIMITER ',')
            """
        else:
            query = f"""
            COPY (
                SELECT * FROM read_parquet('{src_posix}')
                LIMIT {limit}
            )
            TO '{dest_posix}' (HEADER, DELIMITER ',')
            """
        con.execute(query)
        size_mb = dest.stat().st_size / (1024 * 1024)
        print(f"Wrote {dest.name} ({size_mb:.1f} MB)")
finally:
    con.close()

print("\nOpen these for a quick look:")
print(f"  {processed / 'site_parking_windows.csv'}")
print(f"  {processed / 'parking_baseline_preview.csv'}")
print(f"  {processed / 'parking_post_preview.csv'}")


### 6. Sites / streets kept after filtering
Lists which `SiteID`s and streets actually appear in the filtered baseline and post outputs (the ones with CoM 2013–2014 parking overlap).


In [ ]:
import duckdb
from pathlib import Path

from src.config import PROCESSED_DIR

processed = Path(PROCESSED_DIR)
baseline_path = (processed / "parking_baseline.parquet").as_posix()
post_path = (processed / "parking_post.parquet").as_posix()

matched = duckdb.sql(
    f"""
    WITH all_events AS (
        SELECT SiteID, SiteType, StreetInScope, 'baseline' AS period
        FROM read_parquet('{baseline_path}')
        UNION ALL
        SELECT SiteID, SiteType, StreetInScope, 'post' AS period
        FROM read_parquet('{post_path}')
    )
    SELECT
        SiteID,
        SiteType,
        StreetInScope,
        period,
        count(*) AS n_events
    FROM all_events
    GROUP BY SiteID, SiteType, StreetInScope, period
    ORDER BY StreetInScope, SiteID, period
    """
).df()

print("Sites / streets retained after filtering:\n")
print(matched.to_string(index=False))
print(f"\nDistinct sites kept: {matched['SiteID'].nunique()}")
print(f"Distinct streets kept: {matched['StreetInScope'].nunique()}")
print("\nStreets:", ", ".join(sorted(matched["StreetInScope"].dropna().unique())))

matched
